# WIUT Hackathon 2026 · CV Track — team Colab notebook

One notebook for all three of us. Run **Section 0 → 3** every session (≈3–5 min), then jump to your own section.

| Section | Owner | What |
|---|---|---|
| 0–3 | everyone | GPU, Drive, code, **videos from Google Drive links (no download)** |
| 4 | P1 | detection + tracking of all sample videos → cached tracks on Drive, speed, ablation |
| 5 | P2 | scene map with a click tool, traffic-light check, learned flow map |
| 6 | all | labels (`my_labels.json`), clip viewer for exact boundaries |
| 7 | P2 | rules → events → `evaluate.py` (seconds per iteration, thanks to the cache) |
| 8 | P1 | official harness run, 3× time budget, determinism, Part B risk curves |
| 9 | P3 | EDA charts, annotated videos, timelines for the website, demo test |
| 10 | everyone | save + git push |

**Before the first run:** `Runtime → Change runtime type → T4 GPU`.

**Shared Drive folder (do once):** P1 runs Section 2 first — it creates `My Drive/WIUT_CV/` and copies the 4 videos into it *inside Google Drive*.
P1 then shares `WIUT_CV` with P2 and P3 (Editor). P2 and P3 open *Shared with me → WIUT_CV → Add shortcut to Drive → My Drive*.
After that everyone sees the same videos, cache, labels and outputs at `/content/drive/MyDrive/WIUT_CV`, and the GPU work (tracks) is done only once for the whole team.

## 0 · Settings

In [ ]:
TEAM        = "our-team"                 # team name for predictions.json
ME          = "p1"                       # p1 / p2 / p3  (used for your label file)
DRIVE_REL   = "WIUT_CV"                  # folder inside My Drive shared by the team
REPO_URL    = "https://github.com/yelmuratov/Traffic-event-detection-CV-WIUT.git"   # private repo: add Colab secret GH_TOKEN

VIDEO_LINKS = [
    "https://drive.google.com/file/d/1kR9jODA2Wotw4gwkvpRKdqFADNJNc1nS/view?usp=drive_link",
    "https://drive.google.com/file/d/1hp8DYeqtYHSwfM6qAo9FPSRHlpMFrIN_/view?usp=drive_link",
    "https://drive.google.com/file/d/10cHEReCWzO3u-Vk1CnNgHAx6egGy5MwJ/view?usp=drive_link",
    "https://drive.google.com/file/d/1aJ-QsAZVYJtLKHiRvKKeBq1D3GWNobRd/view?usp=drive_link",
]

DRIVE_ROOT    = f"/content/drive/MyDrive/{DRIVE_REL}"
REPO_DIR      = "/content/Traffic-event-detection-CV-WIUT"
LOCAL_SAMPLES = "/content/samples"        # fast local SSD copy of the videos

## 1 · GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nproc; df -h /content | tail -1

## 2 · Google Drive + code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
# Folders are created through the Drive API in Section 3 (creating them here via the mount can race with
# the API and produce duplicate "WIUT_CV" folders). Only list what exists:
print(os.listdir(DRIVE_ROOT) if os.path.exists(DRIVE_ROOT) else f"{DRIVE_ROOT} will be created in Section 3")

**Code.** Preferred: push the repo to GitHub once (P3) and set `REPO_URL`. For a private repo add a Colab secret
`GH_TOKEN` (🔑 icon on the left) with a GitHub token. Until then, upload `wiut-cv.zip` to `My Drive/WIUT_CV/`.

**Starter kit.** Upload the organisers' `run_submission.py`, `evaluate.py` and `examples/` into `My Drive/WIUT_CV/starter_kit/` once;
they are copied into the repo unchanged.

In [ ]:
import os, shutil, subprocess, sys
if not os.path.exists(REPO_DIR):
    if REPO_URL:
        token = ""
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN") or ""
        except Exception:
            pass
        url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        subprocess.run(["git", "clone", "-q", url, REPO_DIR], check=True)
    else:
        subprocess.run(["unzip", "-q", "-o", f"{DRIVE_ROOT}/wiut-cv.zip", "-d", "/content"], check=True)

# starter kit files (must stay unchanged)
kit = f"{DRIVE_ROOT}/starter_kit"
for name in ["run_submission.py", "evaluate.py", "examples"]:
    src, dst = f"{kit}/{name}", f"{REPO_DIR}/{name}"
    if os.path.exists(src) and not os.path.exists(dst):
        (shutil.copytree if os.path.isdir(src) else shutil.copy)(src, dst)
    print(("✓ " if os.path.exists(dst) else "✗ missing ") + name)

os.environ["WIUT_CACHE"] = f"{DRIVE_ROOT}/cache"  # created in Section 3   # tracks are cached on Drive -> GPU work done once for the team
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
!pip install -q -r requirements.txt gdown
!bash weights/download.sh

## 3 · Videos: Google Drive link → your Drive → Colab disk (no download)

The four sample videos are ~5–6 GB. Downloading them (browser or `gdown`) is slow and often fails with
*"Too many users have viewed or downloaded this file"*. Instead we:

1. **copy them server-side** with the Drive API (`files.copy`) into `My Drive/WIUT_CV/samples/` — nothing is downloaded, it takes ~1 min per file and happens only once for the team;
2. read them through the Drive mount and **copy to the local SSD** `/content/samples` (fast, ~100–200 MB/s) for random access while decoding;
3. **verify** each file with `ffprobe` (duration should be 5:40, 5:18, 5:18, 2:07).

You will be asked once to allow Colab to access your Google account (needed for the Drive API).

In [ ]:
from tools import colab as cb
drive_paths = cb.import_videos(VIDEO_LINKS, f"{DRIVE_REL}/samples")
VIDEOS = cb.copy_local(drive_paths, LOCAL_SAMPLES)
cb.verify(VIDEOS)
for sub in ["cache", "labels", "outputs", "starter_kit"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

**Fallbacks if the copy fails** (e.g. the owner disabled copying):
- run `VIDEOS = cb.download_with_gdown(VIDEO_LINKS, LOCAL_SAMPLES)` (direct download, may hit the quota), or
- open each link in the browser → ⋮ → **Add shortcut to Drive** → `My Drive/WIUT_CV/samples`, then run the cell below.

In [ ]:
# Use whatever is in the Drive samples folder (after manual shortcuts), then copy locally
import glob
VIDEOS = cb.copy_local(sorted(glob.glob(f"{DRIVE_ROOT}/samples/*.mp4")), LOCAL_SAMPLES)
cb.verify(VIDEOS)

In [ ]:
# Quick look: metadata + a frame from each video
import pandas as pd
from src.video import probe, read_frame_at
meta = pd.DataFrame([probe(p).to_dict() for p in VIDEOS]).drop(columns="path")
display(meta)
for p in VIDEOS:
    print(os.path.basename(p)); cb.show(read_frame_at(p, 5.0), width=900)

In [ ]:
# Imports used by all sections below (run this even if you skip to your own section)
import glob, json, time, logging
import numpy as np, matplotlib.pyplot as plt
from src import config
from src.scene import load_scene
from src.tracker import analyse_video
logging.getLogger("wiut").setLevel(logging.INFO)

## 4 · P1 — detection + tracking (GPU)

One decoding pass per video: YOLO11m @1280 on every 2nd frame + ByteTrack. Results are cached in
`WIUT_CV/cache/` keyed by video + detector settings, so P2/P3 never have to run the GPU part again unless
`config.DETECTOR` / `config.TRACKER` change. Budget reference: Part A + B together must stay well under **3× video length**.

In [ ]:
import time, logging
logging.getLogger("wiut").setLevel(logging.INFO)
from src import config
from src.scene import load_scene
from src.tracker import analyse_video
print("device:", config.DEVICE, "| detector:", config.DETECTOR)
speed = []
for p in VIDEOS:
    m = probe(p); t0 = time.time()
    tracks, _ = analyse_video(m, load_scene(m.width, m.height), progress=True)
    dt = time.time() - t0
    speed.append({"video": m.name, "rows": len(tracks), "tracks": len(set(tracks[:, 2])) if len(tracks) else 0,
                  "seconds": round(dt, 1), "x_realtime": round(dt / m.duration, 2)})
display(pd.DataFrame(speed))   # a cached video shows ~0 s

**Ablation (website extra credit):** detector × input size on the first 60 s of one video — speed and detections per frame.

In [ ]:
import torch
from ultralytics import YOLO
from src.video import iter_frames
vid = VIDEOS[0]; m = probe(vid); end = int(60 * m.fps)

t0 = time.time(); n = 0
for _ in iter_frames(vid, stride=2, max_width=1920, end=end): n += 1
decode_fps = n / (time.time() - t0)
print(f"decode only: {decode_fps:.1f} processed frames/s (stride 2)")

rows = []
for w, sz in [("yolo11s.pt", 960), ("yolo11m.pt", 960), ("yolo11m.pt", 1280), ("yolo26m.pt", 1280)]:
    model = YOLO(str(config.WEIGHTS_DIR / w) if (config.WEIGHTS_DIR / w).exists() else w)
    counts, batch, t_inf, n = {}, [], 0.0, 0
    for fi, t, f, s in iter_frames(vid, stride=2, max_width=1920, end=end):
        batch.append(f)
        if len(batch) == 8:
            ti = time.time()
            res = model.predict(batch, imgsz=sz, half=True, device=0, classes=config.DET_CLASSES, conf=0.25, verbose=False)
            torch.cuda.synchronize(); t_inf += time.time() - ti
            for r in res:
                for c in r.boxes.cls.int().tolist():
                    counts[c] = counts.get(c, 0) + 1
            n += len(batch); batch = []
    rows.append({"model": w, "imgsz": sz, "infer_ms/frame": round(1000 * t_inf / n, 1),
                 **{f"{k}/frame": round(v / n, 2) for k, v in sorted(counts.items())}})
abl = pd.DataFrame(rows).rename(columns=lambda c: c.replace("0/", "person/").replace("2/", "car/").replace("7/", "truck/"))
display(abl); abl.to_csv(f"{DRIVE_ROOT}/outputs/ablation_detector.csv", index=False)

## 5 · P2 — scene map (click tool)

Draw the fixed layout once on a reference frame; it is saved to `scene/scene_map.json` (commit it) and to Drive.
Every cell opens a canvas: click points, then **Done**. Re-run a cell to add another item; `undo("lanes")` removes the last one.
Pick a reference time where the road is fairly empty and the traffic light is visible.

In [ ]:
import json
from src.scene import Scene, EMPTY_SCENE
REF_VIDEO, REF_T = VIDEOS[0], 10.0
ref = read_frame_at(REF_VIDEO, REF_T)
scene = json.load(open(config.SCENE_PATH)) if os.path.exists(config.SCENE_PATH) else dict(EMPTY_SCENE)
for k, v in EMPTY_SCENE.items(): scene.setdefault(k, v if not isinstance(v, (list, dict)) else type(v)())
scene["size"] = [ref.shape[1], ref.shape[0]]

def save_scene(show=True):
    json.dump(scene, open(config.SCENE_PATH, "w"), indent=1)
    shutil.copy(config.SCENE_PATH, f"{DRIVE_ROOT}/scene_map.json")
    if show: cb.show(Scene(scene, ref.shape[1], ref.shape[0]).draw(ref))

def undo(key):
    if scene[key]: scene[key].pop(); save_scene()

save_scene()

In [ ]:
# Carriageway (road surface only - NOT sidewalks). Several polygons allowed.
scene["road"].append(cb.pick(ref, "Road / carriageway polygon")); save_scene()

In [ ]:
# Pedestrian crossings (zebra)
scene["crosswalks"].append(cb.pick(ref, "Crosswalk polygon")); save_scene()

In [ ]:
# Lanes: polygon + allowed driving direction (tail -> head). `group` = one direction of traffic (used by congestion).
LANE_NAME, GROUP = "lane_1", "dir_A"
poly  = cb.pick(ref, f"{LANE_NAME}: lane polygon")
arrow = cb.pick(ref, f"{LANE_NAME}: allowed direction (click tail, then head)", "line")
scene["lanes"].append({"name": LANE_NAME, "polygon": poly, "direction": cb.direction_from(arrow), "group": GROUP})
save_scene()

In [ ]:
# Traffic light: first a box around the whole light (zoom), then the RED and GREEN lamp boxes inside the zoom.
SIGNAL = "sig_1"
zoom  = cb.box_from(cb.pick(ref, "Box around the traffic light (for zoom)", "box"))
red   = cb.box_from(cb.pick(ref, "RED lamp", "box", crop=zoom))
green = cb.box_from(cb.pick(ref, "GREEN lamp", "box", crop=zoom))
scene["signals"][SIGNAL] = {"red_roi": red, "green_roi": green}; save_scene()

In [ ]:
# Stop line controlled by SIGNAL: the line, the approach direction, and the intersection area beyond it.
line  = cb.pick(ref, "Stop line (2 points)", "line")
arrow = cb.pick(ref, "Direction of traffic crossing the line (tail, head)", "line")
inter = cb.pick(ref, "Intersection area beyond the stop line (polygon)")
scene["stop_lines"].append({"name": f"stop_{len(scene['stop_lines'])+1}", "line": line,
                            "direction": cb.direction_from(arrow), "signal": SIGNAL, "intersection": inter})
save_scene()

In [ ]:
# Solid lane markings (polyline along the paint)
scene["solid_lines"].append(cb.pick(ref, "Solid line (click along it)", "polyline")); save_scene()

In [ ]:
# Queue zones (stopping here at a red light is normal) and ignore zones (parking, sidewalk, far background)
scene["queue_zones"].append(cb.pick(ref, "Queue zone before the stop line")); save_scene()
# scene["ignore_zones"].append(cb.pick(ref, "Ignore zone")); save_scene()

In [ ]:
# Forbidden movement: vehicle goes from zone A to zone B where that turn is prohibited (use signs/markings in camera.md)
frm = cb.pick(ref, "Forbidden movement: ENTRY zone")
to  = cb.pick(ref, "Forbidden movement: EXIT zone")
scene["forbidden_movements"].append({"name": f"move_{len(scene['forbidden_movements'])+1}", "from": frm, "to": to,
                                     "label": "illegal_turn"})   # or "illegal_u_turn"
save_scene()

In [ ]:
# Zones where any U-turn is illegal
scene["no_u_turn_zones"].append(cb.pick(ref, "No-U-turn zone")); save_scene()

**Check the traffic-light reader** — the state should flip red/green in a regular cycle. If it is mostly `-1` (unknown), redraw the lamp boxes tighter.

In [ ]:
import matplotlib.pyplot as plt
from src.scene import smooth_states
for p in VIDEOS[:2]:
    m = probe(p); sc = load_scene(m.width, m.height)
    _, sig = analyse_video(m, sc, want_tracks=False)
    for name, (ts, st) in sig.items():
        plt.figure(figsize=(14, 1.8)); plt.plot(ts, st, lw=0.6, label="raw"); plt.plot(ts, smooth_states(st, 13), lw=1.5, label="smoothed")
        plt.yticks([-1, 0, 1], ["unknown", "green", "red"]); plt.title(f"{m.name} · {name}"); plt.legend(); plt.show()

**Learned flow map** — dominant driving direction per cell from all sample tracks (green = one-way, orange = mixed). Used for `wrong_way` when no lanes are drawn, and a good guide for drawing lanes.

In [ ]:
from src.tracks import build_table
from src.scene import FlowMap
tables = []
for p in VIDEOS:
    m = probe(p); tr, _ = analyse_video(m, None); tables.append(build_table(tr, m.fps))
flow = FlowMap.build(pd.concat(tables), m.width, m.height)
flow.save(); cb.show(flow.draw(ref))

## 6 · Labels (everyone) — our dev set

Split: **P1 → video 1, P2 → video 2, P3 → videos 3 + 4** (4 is only 2 min). Follow the start/end conventions in the task PDF exactly —
at tIoU 0.7 a boundary that is 1 s off on a 5 s event already fails.
Edit your file `WIUT_CV/labels/labels_<you>.json` (double-click it in the Colab file browser, or edit on your laptop):
`[start_sec, end_sec, "label"]` in each video's `events` list.

In [ ]:
from tools.labels import skeleton, merge, check, summary
my_file = f"{DRIVE_ROOT}/labels/labels_{ME}.json"
if not os.path.exists(my_file):
    json.dump(skeleton(LOCAL_SAMPLES), open(my_file, "w"), indent=1)
print("edit:", my_file)

In [ ]:
# Watch a short window with the timestamp burned in to set exact boundaries
cb.play_clip(VIDEOS[0], t0=10.0, t1=20.0)

In [ ]:
# Merge everyone's labels -> my_labels.json (repo + Drive)
files = sorted(glob.glob(f"{DRIVE_ROOT}/labels/labels_p*.json"))
labels = merge(files)
print(check(labels) or "labels OK", summary(labels))
json.dump(labels, open("my_labels.json", "w"), indent=1); shutil.copy("my_labels.json", f"{DRIVE_ROOT}/labels/my_labels.json")

## 7 · P2 — rules → events → score (fast loop)

Tracks come from the cache, so one iteration over all videos takes seconds. Tune thresholds in `src/config.py`
(open it in the Colab file browser), then re-run this cell. Classes that never score well on our labels should be switched off in `config.ENABLED`.

In [ ]:
import importlib
def reload_all():
    import src.config, src.utils, src.video, src.scene, src.tracker, src.tracks, src.rules, src.postprocess, src.pipeline, src.risk
    for mod in [src.config, src.utils, src.video, src.scene, src.tracker, src.tracks, src.rules, src.postprocess, src.pipeline, src.risk]:
        importlib.reload(mod)
reload_all()
from src.pipeline import detect_events

preds, DEBUG = {"team": TEAM, "videos": {}}, {}
for p in VIDEOS:
    m = probe(p)
    ev, raw, ctx = detect_events(p, return_raw=True)
    DEBUG[m.name] = (raw, ctx)
    preds["videos"][m.name] = {"events": ev, "risk": [[round(i / m.fps, 3), 0.0] for i in range(m.n_frames)]}
    print(f"{m.name}: {len(ev)} events", pd.Series([e[2] for e in ev]).value_counts().to_dict())
json.dump(preds, open("/content/pred_rules_only.json", "w"))
!python evaluate.py --pred /content/pred_rules_only.json --validate-only
if os.path.exists("my_labels.json"):
    !python evaluate.py --pred /content/pred_rules_only.json --gt my_labels.json --per-video

In [ ]:
# Inspect one video: predicted vs labelled events on a timeline
from tools.render import timeline_png
from IPython.display import Image
name = os.path.basename(VIDEOS[0])
gt = json.load(open("my_labels.json")).get(name, {}).get("events") if os.path.exists("my_labels.json") else None
timeline_png(preds["videos"][name]["events"], probe(VIDEOS[0]).duration, "/content/tl.png", gt=gt, title=name)
display(Image("/content/tl.png"))

## 8 · P1 — official harness run: time budget, determinism, Part B

This is exactly what the organisers run (cache disabled, full Part A + Part B). On Colab's T4 the total must be well under
`3 × total video length`. Colab has only 2 CPU cores (the judges' machine has 8), so decoding here is a pessimistic estimate.

In [ ]:
total = sum(probe(p).duration for p in VIDEOS)
t0 = time.time()
!env -u WIUT_CACHE python run_submission.py --videos {LOCAL_SAMPLES} --out predictions_samples.json --team {TEAM}
wall = time.time() - t0
print(f"wall {wall:.0f}s for {total:.0f}s of video = {wall/total:.2f}x  (limit 3.0x per video)")
!python evaluate.py --pred predictions_samples.json --validate-only
if os.path.exists("my_labels.json"):
    !python evaluate.py --pred predictions_samples.json --gt my_labels.json --per-video
shutil.copy("predictions_samples.json", f"{DRIVE_ROOT}/outputs/predictions_samples.json")

In [ ]:
# Determinism: a second run must give the same JSON (up to float noise)
!env -u WIUT_CACHE python run_submission.py --videos {LOCAL_SAMPLES} --out /content/pred_run2.json --team {TEAM}
import numpy as np
a, b = json.load(open("predictions_samples.json")), json.load(open("/content/pred_run2.json"))
for k in a["videos"]:
    same_ev = a["videos"][k]["events"] == b["videos"][k]["events"]
    ra, rb = np.array(a["videos"][k]["risk"]), np.array(b["videos"][k]["risk"])
    print(k, "events identical:", same_ev, "| max risk diff:", float(np.abs(ra - rb).max()) if len(ra) else 0)

In [ ]:
# Part B risk curves (orange = the 5 s before each labelled accident, where the score should rise above 0.5)
labels = json.load(open("my_labels.json")) if os.path.exists("my_labels.json") else {}
for k, v in json.load(open("predictions_samples.json"))["videos"].items():
    r = np.array(v["risk"])
    plt.figure(figsize=(14, 2)); plt.plot(r[:, 0], r[:, 1], lw=0.8, color="#c0392b"); plt.axhline(0.5, ls="--", c="grey")
    for s, e, l in labels.get(k, {}).get("events", []):
        if l == "accident": plt.axvspan(s - 5, s, color="orange", alpha=0.3)
        if l == "near_miss": plt.axvspan(s - 5, e, color="grey", alpha=0.2)
    plt.ylim(0, 1); plt.title(k); plt.show()

In [ ]:
# Calibrate the final gain offline (score = clip(gain * ema)); pick the best Score B, then set config.RISK["gain"]
base = json.load(open("predictions_samples.json"))
g0 = config.RISK["gain"]
for g in [0.5, 0.75, 1.0, 1.5, 2.0, 3.0]:
    d = json.loads(json.dumps(base))
    for v in d["videos"].values():
        v["risk"] = [[t, float(min(1.0, s * g / g0))] for t, s in v["risk"]]
    json.dump(d, open("/content/pred_gain.json", "w"))
    print(f"--- gain {g}")
    !python evaluate.py --pred /content/pred_gain.json --gt my_labels.json | grep -i -E "score b|model"

## 9 · P3 — website material (EDA, annotated videos, timelines) and demo

In [ ]:
!python -m tools.eda {LOCAL_SAMPLES} {DRIVE_ROOT}/outputs/eda

In [ ]:
# Annotated videos (boxes, IDs, scene overlay, event banners, risk bar) + timeline PNG + events JSON per video
for p in VIDEOS:
    !python -m tools.render {p} {DRIVE_ROOT}/outputs/render --pred predictions_samples.json --gt my_labels.json
!ls -lh {DRIVE_ROOT}/outputs/render

In [ ]:
# Try the live demo from Colab (public *.gradio.live link, CPU settings). Stop the cell to end it.
!pip install -q gradio
!WIUT_DEMO=1 GRADIO_SHARE=True python demo/app.py

## 10 · Save and push

In [ ]:
# scene map, flow map, labels and predictions belong in the repo; videos and cache do not (.gitignore)
!git status --short
# from google.colab import userdata
# token = userdata.get("GH_TOKEN")
# !git config user.name "{ME}" && git config user.email "{ME}@team"
# !git add -A && git commit -m "scene map + labels + predictions" && git push https://{token}@github.com/yelmuratov/Traffic-event-detection-CV-WIUT.git HEAD:main